# 🔍 TruthLens - Document Fraud Detection
## Training Notebook - EfficientNet + ViT Ensemble

**Student:** Shravani R S  
**Institution:** IIIT Dharwad  
**Program:** M.Tech - Data Science & AI  

### Dataset
- 2,880 images across 7 Indian document types
- Real documents: ID cards, certificates, bank statements, invoices, payslips, experience letters, marksheets
- Fake documents: Tampered versions with blur, brightness patches, erasure, contrast manipulation
- Split: 70% train / 15% val / 15% test

### Model Architecture
- EfficientNet-B3 (texture & artifact detection)
- ViT-Small (layout & semantic analysis)
- Weighted ensemble combination
- Target accuracy: 95%+

## Step 1: Install Dependencies

In [ ]:
!pip install timm==1.0.3 -q
print('✅ Dependencies installed')

## Step 2: Upload & Extract Dataset
Upload `training_data.zip` from your PC when prompted.

In [ ]:
from google.colab import files
import zipfile, os

print('Upload training_data.zip from your PC...')
uploaded = files.upload()

zip_name = list(uploaded.keys())[0]
print(f'Extracting {zip_name}...')
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('/content/training_data')

# Count images
for split in ['train', 'val', 'test']:
    for label in ['real', 'fake']:
        path = f'/content/training_data/{split}/{label}'
        if os.path.exists(path):
            count = len(os.listdir(path))
            print(f'  {split}/{label}: {count} images')

print('✅ Dataset ready!')

## Step 3: Imports & Configuration

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import timm
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import time, os, copy

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

# Configuration
CONFIG = {
    'data_dir': '/content/training_data',
    'batch_size': 16,
    'num_epochs': 25,
    'learning_rate': 1e-4,
    'weight_decay': 0.01,
    'img_size': 224,
    'num_classes': 2,
    'patience': 7,
}
print('\n✅ Configuration set')
print(f'  Epochs: {CONFIG["num_epochs"]}')
print(f'  Batch size: {CONFIG["batch_size"]}')
print(f'  Learning rate: {CONFIG["learning_rate"]}')

## Step 4: Data Augmentation & Loaders

In [ ]:
# Strong augmentation for training
train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
    transforms.RandomGrayscale(p=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.1),
])

# No augmentation for val/test
val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

# Datasets
train_dataset = datasets.ImageFolder(os.path.join(CONFIG['data_dir'], 'train'), transform=train_transform)
val_dataset   = datasets.ImageFolder(os.path.join(CONFIG['data_dir'], 'val'),   transform=val_transform)
test_dataset  = datasets.ImageFolder(os.path.join(CONFIG['data_dir'], 'test'),  transform=val_transform)

# DataLoaders
train_loader = DataLoader(train_dataset, batch_size=CONFIG['batch_size'], shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=CONFIG['batch_size'], shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=CONFIG['batch_size'], shuffle=False, num_workers=2, pin_memory=True)

print(f'Classes: {train_dataset.classes}')
print(f'Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}')
print('✅ Data loaders ready')

## Step 5: Model Architecture - EfficientNet + ViT Ensemble

In [ ]:
class TruthLensEnsemble(nn.Module):
    """
    TruthLens Ensemble Model
    Combines EfficientNet-B3 (CNN) and ViT-Small (Transformer)
    EfficientNet: detects texture artifacts, printing anomalies, compression artifacts
    ViT: detects layout inconsistencies, semantic patterns, structural anomalies
    """
    def __init__(self, num_classes=2):
        super(TruthLensEnsemble, self).__init__()

        # EfficientNet-B3 branch
        self.efficientnet = timm.create_model('efficientnet_b3', pretrained=True)
        eff_features = self.efficientnet.classifier.in_features
        self.efficientnet.classifier = nn.Identity()

        # ViT-Small branch
        self.vit = timm.create_model('vit_small_patch16_224', pretrained=True)
        vit_features = self.vit.head.in_features
        self.vit.head = nn.Identity()

        # Fusion layers
        combined = eff_features + vit_features
        self.fusion = nn.Sequential(
            nn.Linear(combined, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(512, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        eff_out = self.efficientnet(x)
        vit_out = self.vit(x)
        combined = torch.cat([eff_out, vit_out], dim=1)
        return self.fusion(combined)


# Initialize model
model = TruthLensEnsemble(num_classes=CONFIG['num_classes']).to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total parameters:     {total_params:,}')
print(f'Trainable parameters: {trainable_params:,}')
print('✅ TruthLens Ensemble Model created')

## Step 6: Training Setup

In [ ]:
# Loss function
criterion = nn.CrossEntropyLoss()

# Optimizer - different LR for pretrained vs new layers
optimizer = optim.AdamW([
    {'params': model.efficientnet.parameters(), 'lr': CONFIG['learning_rate'] * 0.1},
    {'params': model.vit.parameters(),          'lr': CONFIG['learning_rate'] * 0.1},
    {'params': model.fusion.parameters(),       'lr': CONFIG['learning_rate']},
], weight_decay=CONFIG['weight_decay'])

# Scheduler
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CONFIG['num_epochs'], eta_min=1e-6)

print('✅ Optimizer: AdamW with layer-wise learning rates')
print('✅ Scheduler: CosineAnnealingLR')
print('✅ Loss: CrossEntropyLoss')

## Step 7: Training Loop

In [ ]:
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for inputs, labels in loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        _, predicted = outputs.max(1)
        correct += predicted.eq(labels).sum().item()
        total += labels.size(0)
    return total_loss / len(loader), 100. * correct / total


def val_epoch(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            total_loss += loss.item()
            _, predicted = outputs.max(1)
            correct += predicted.eq(labels).sum().item()
            total += labels.size(0)
    return total_loss / len(loader), 100. * correct / total


# Training loop
best_val_acc = 0
best_model_wts = copy.deepcopy(model.state_dict())
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
patience_counter = 0
start_time = time.time()

print('='*65)
print('STARTING TRAINING - TruthLens Ensemble')
print('='*65)

for epoch in range(CONFIG['num_epochs']):
    t0 = time.time()

    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
    val_loss,   val_acc   = val_epoch(model, val_loader, criterion, device)
    scheduler.step()

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)

    elapsed = time.time() - t0
    marker = ' ← BEST' if val_acc > best_val_acc else ''

    print(f'Epoch [{epoch+1:02d}/{CONFIG["num_epochs"]}] '
          f'Train: {train_acc:.2f}% | Val: {val_acc:.2f}% | '
          f'Loss: {val_loss:.4f} | Time: {elapsed:.0f}s{marker}')

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_model_wts = copy.deepcopy(model.state_dict())
        torch.save(model.state_dict(), 'best_truthlens_model.pth')
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= CONFIG['patience']:
            print(f'\nEarly stopping at epoch {epoch+1}')
            break

total_time = (time.time() - start_time) / 60
print('='*65)
print(f'Training complete! Best Val Accuracy: {best_val_acc:.2f}%')
print(f'Total time: {total_time:.1f} minutes')
print('='*65)

## Step 8: Plot Training Curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

epochs_ran = range(1, len(history['train_acc']) + 1)

ax1.plot(epochs_ran, history['train_acc'], 'b-o', label='Train Accuracy', markersize=4)
ax1.plot(epochs_ran, history['val_acc'],   'r-o', label='Val Accuracy',   markersize=4)
ax1.axhline(y=95, color='g', linestyle='--', label='95% Target')
ax1.set_title('TruthLens - Accuracy Curves', fontsize=13, fontweight='bold')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy (%)')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(epochs_ran, history['train_loss'], 'b-o', label='Train Loss', markersize=4)
ax2.plot(epochs_ran, history['val_loss'],   'r-o', label='Val Loss',   markersize=4)
ax2.set_title('TruthLens - Loss Curves', fontsize=13, fontweight='bold')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Training curves saved')

## Step 9: Final Evaluation on Test Set

In [ ]:
# Load best model
model.load_state_dict(best_model_wts)
model.eval()

all_preds, all_labels = [], []
correct, total = 0, 0

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        _, predicted = outputs.max(1)
        correct += predicted.eq(labels).sum().item()
        total += labels.size(0)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

test_acc = 100. * correct / total

print('='*55)
print('FINAL TEST RESULTS - TruthLens')
print('='*55)
print(f'Test Accuracy:  {test_acc:.2f}%')
print(f'Best Val Acc:   {best_val_acc:.2f}%')
print('='*55)
print('\nClassification Report:')
print(classification_report(all_labels, all_preds, target_names=['Fake', 'Real']))

## Step 10: Confusion Matrix

In [ ]:
cm = confusion_matrix(all_labels, all_preds)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Fake', 'Real'],
            yticklabels=['Fake', 'Real'],
            linewidths=0.5)
plt.title(f'TruthLens Confusion Matrix\nTest Accuracy: {test_acc:.2f}%',
          fontsize=13, fontweight='bold')
plt.ylabel('True Label', fontsize=11)
plt.xlabel('Predicted Label', fontsize=11)
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Confusion matrix saved')

## Step 11: Download Trained Model

In [ ]:
from google.colab import files

# Download model and results
print('Downloading trained model and results...')
files.download('best_truthlens_model.pth')
files.download('training_curves.png')
files.download('confusion_matrix.png')

print('\n' + '='*55)
print('✅ ALL DONE!')
print('='*55)
print(f'Best Validation Accuracy: {best_val_acc:.2f}%')
print(f'Final Test Accuracy:      {test_acc:.2f}%')
print('\nFiles downloaded:')
print('  best_truthlens_model.pth  ← Save this to 02_models/')
print('  training_curves.png       ← Use in thesis/paper')
print('  confusion_matrix.png      ← Use in thesis/paper')